In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import frangi
from skimage.morphology import skeletonize

# 1. Define the file names from your directory
file_list = [
    'image0000a.png', 'image0001a.png', 'image0001b.png', 'image0002a.png',
    'image0002b.png', 'image0003a.png', 'image0003b.png', 'image0004a.png',
    'image0004b.png', 'image0005a.png', 'image0005b.png', 'image0006a.png',
    'image0006b.png', 'image0007a.png', 'image0007b.png', 'image0008a.png',
    'image0008b.png', 'image0009a.png', 'image0009b.png', 'image0000b.png' ]

# 2. Import and Preprocess (Grayscale conversion)
synthetic_images = []
for fname in file_list[6:9]:
    img = plt.imread(f'{fname}')
    if img.ndim == 3: # Handle RGB/RGBA to grayscale conversion
        img = img[..., :3].dot([0.2989, 0.5870, 0.1140]) # Luminosity method
    synthetic_images.append(img)

# Load the Ground Truth mask from your directory
gt_mask =plt.imread(f'{fname}')
if gt_mask.ndim == 3:
    gt_mask = gt_mask[..., :3].dot([0.2989, 0.5870, 0.1140])
ground_truth_images = [gt_mask] * len(synthetic_images)

# 3. Your Vessel Segmentation Pipeline Class
class VesselSegmentationPipeline:
    def __init__(self, frangi_scales=(10.0, 50.0), frangi_step=0.5,
                 morphology_kernel_size=3, vessel_threshold=0.01):
        self.frangi_scales = np.arange(frangi_scales[0], frangi_scales[1] + frangi_step, frangi_step)
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morphology_kernel_size, morphology_kernel_size))
        self.vessel_threshold = vessel_threshold

    def segment(self, image):
        # Apply Frangi filter for vesselness detection
        vesselness = frangi(image, sigmas=self.frangi_scales, black_ridges=False)
        binary = (vesselness > self.vessel_threshold).astype(np.uint8) * 255

        # Morphological operations to clean the mask
        opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, self.kernel)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, self.kernel)

        # Keep only the largest connected component (main vessel tree)
        num_labels, labels = cv2.connectedComponents(closed)
        if num_labels > 1:
            vessel_mask = (labels == (1 + np.argmax(np.bincount(labels.ravel())[1:]))).astype(np.uint8) * 255
        else:
            vessel_mask = closed

        skeleton = skeletonize(vessel_mask > 0).astype(np.uint8) * 255
        return vessel_mask, skeleton, vesselness

def dice_score(pred, ground_truth):
    intersection = np.logical_and(pred, ground_truth).sum()
    return 2.0 * intersection / (pred.sum() + ground_truth.sum() + 1e-8)

# 4. Initialize and Run Pipeline
pipeline = VesselSegmentationPipeline(vessel_threshold=0.06)
sample_img = synthetic_images[0]
mask, skeleton, vesselness = pipeline.segment(sample_img)

# Visualization
num_images = len(synthetic_images)
fig, axes = plt.subplots(num_images, 4, figsize=(20, 5 * num_images))
if num_images == 1:
    axes = np.expand_dims(axes, axis=0)

for i, img in enumerate(synthetic_images):
    # Run the pipeline for each image
    mask, skeleton, vesselness = pipeline.segment(img)

    axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title(f'Image {i}: Original')
    axes[i, 1].imshow(vesselness, cmap='hot'); axes[i, 1].set_title(f'Image {i}: Frangi Vesselness')
    axes[i, 2].imshow(mask, cmap='gray'); axes[i, 2].set_title(f'Image {i}: Morphological Mask')
    axes[i, 3].imshow(skeleton, cmap='magma'); axes[i, 3].set_title(f'Image {i}: Centrelines')

    for ax in axes[i]:
            ax.axis('off')

plt.show()

# 5. Sensitivity Analysis
thresholds = [1e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 6e-2]
scores = []

for t in thresholds:
    pipeline.vessel_threshold = t
    mask, _, _ = pipeline.segment(sample_img)
    scores.append(dice_score(mask > 0, ground_truth_images[0] > 0))

plt.plot(thresholds, scores, marker='o')
plt.title("Sensitivity: Threshold vs. Dice Score")
plt.xlabel("Frangi Threshold"); plt.ylabel("Dice Score")
plt.grid(True)
plt.show()

# Print Results
print("\nSensitivity Analysis Results:")
print(f"{'Threshold':<15} | {'Dice Score':<10}")
print("-" * 30)
for t, s in zip(thresholds, scores):
    print(f"{t:<15.6f} | {s:.4f}")

best_idx = np.argmax(scores)
print("-" * 30)
print(f"Best Configuration: Threshold={thresholds[best_idx]} -> Dice={scores[best_idx]:.4f}")

# Task
```python
# 1. Define the file names from your directory
file_list = [
    'image0000a.png', 'image0001a.png', 'image0001b.png', 'image0002a.png',
    'image0003a.png', 'image0003b.png', 'image0004a.png', 'image0005a.png',
    'image0005b.png', 'image0006a.png', 'image0007a.png', 'image0007b.png',
    'image0008a.png', 'image0009a.png', 'image0009b.png' ]

# 2. Import and Preprocess (Grayscale conversion)
synthetic_images = []
for fname in file_list[6:9]:
    img = plt.imread(f'{fname}')
    if img.ndim == 3: # Handle RGB/RGBA to grayscale conversion
        img = img[..., :3].dot([0.2989, 0.5870, 0.1140]) # Luminosity method
    synthetic_images.append(img)

# Load the Ground Truth mask from your directory
gt_mask =plt.imread(f'{fname}')
if gt_mask.ndim == 3:
    gt_mask = gt_mask[..., :3].dot([0.2989, 0.5870, 0.1140])
ground_truth_images = [gt_mask] * len(synthetic_images)

# 3. Your Vessel Segmentation Pipeline Class
class VesselSegmentationPipeline:
    def __init__(self, frangi_scales=(10.0, 50.0), frangi_step=0.5,
                 morphology_kernel_size=3, vessel_threshold=0.01):
        self.frangi_scales = np.arange(frangi_scales[0], frangi_scales[1] + frangi_step, frangi_step)
        self.kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morphology_kernel_size, morphology_kernel_size))
        self.vessel_threshold = vessel_threshold

    def segment(self, image):
        # Apply Frangi filter for vesselness detection
        vesselness = frangi(image, sigmas=self.frangi_scales, black_ridges=False)
        binary = (vesselness > self.vessel_threshold).astype(np.uint8) * 255

        # Morphological operations to clean the mask
        opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, self.kernel)
        closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, self.kernel)

        # Keep only the largest connected component (main vessel tree)
        num_labels, labels = cv2.connectedComponents(closed)
        if num_labels > 1:
            vessel_mask = (labels == (1 + np.argmax(np.bincount(labels.ravel())[1:]))).astype(np.uint8) * 255
        else:
            vessel_mask = closed

        skeleton = skeletonize(vessel_mask > 0).astype(np.uint8) * 255
        return vessel_mask, skeleton, vesselness

def dice_score(pred, ground_truth):
    intersection = np.logical_and(pred, ground_truth).sum()
    return 2.0 * intersection / (pred.sum() + ground_truth.sum() + 1e-8)

# 4. Initialize and Run Pipeline
pipeline = VesselSegmentationPipeline(vessel_threshold=0.06)
sample_img = synthetic_images[0]
mask, skeleton, vesselness = pipeline.segment(sample_img)

# Visualization
num_images = len(synthetic_images)
fig, axes = plt.subplots(num_images, 4, figsize=(20, 5 * num_images))
if num_images == 1:
    axes = np.expand_dims(axes, axis=0)

for i, img in enumerate(synthetic_images):
    # Run the pipeline for each image
    mask, skeleton, vesselness = pipeline.segment(img)

    # Corrected: Use 'img' for the original image and ensure all visualization steps are inside the loop
    axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title(f'Image {i}: Original')
    axes[i, 1].imshow(vesselness, cmap='hot'); axes[i, 1].set_title(f'Image {i}: Frangi Vesselness')
    axes[i, 2].imshow(mask, cmap='gray'); axes[i, 2].set_title('3. Morphological Mask')
    axes[i, 3].imshow(skeleton, cmap='magma'); axes[i, 3].set_title(f'Image {i}: centrelines')

    for ax in axes[i]:
            ax.axis('off')

plt.tight_layout() # Added to prevent labels from overlapping
plt.show()

# 5. Sensitivity Analysis
thresholds = [1e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 6e-2]
scores = []

for t in thresholds:
    pipeline.vessel_threshold = t
    mask, _, _ = pipeline.segment(sample_img)
    scores.append(dice_score(mask > 0, ground_truth_images[0] > 0))

plt.plot(thresholds, scores, marker='o')
plt.title("Sensitivity: Threshold vs. Dice Score")
plt.xlabel("Frangi Threshold"); plt.ylabel("Dice Score")
plt.grid(True)
plt.show()

# Print Results
print("\nSensitivity Analysis Results:")
print(f"{'Threshold':<15} | {'Dice Score':<10}")
print("-" * 30)
for t, s in zip(thresholds, scores):
    print(f"{t:<15.6f} | {s:.4f}")

best_idx = np.argmax(scores)
print("-" * 30)
print(f"Best Configuration: Threshold={thresholds[best_idx]} -> Dice={scores[best_idx]:.4f}")
```

## Correct Visualization Loop

### Subtask:
Modify the existing visualization loop to display the correct original image for each row.


## Final Task

### Subtask:
Confirm that the visualization now correctly displays the original image, vesselness, morphological mask, and skeleton for each synthetic image loaded, and that the sensitivity analysis remains as previously computed.


## Summary:

### Q&A
The visualization now correctly displays the original image, vesselness, morphological mask, and skeleton for each synthetic image loaded. The sensitivity analysis remained as previously computed and was not affected by the visualization correction.

### Data Analysis Key Findings
*   The visualization loop correctly displayed the original image for each row, as the code already used the `img` variable, which represents the current image in the iteration. No modification was needed.
*   The sensitivity analysis identified the optimal Frangi threshold for the given `sample_img` (the first image in the `synthetic_images` list).
*   The highest Dice score achieved was 0.4806 with a Frangi threshold of 1e-06.
*   The Dice scores varied significantly with different thresholds, ranging from a minimum of 0.0000 (at thresholds 5e-04 and 1e-02) to the maximum of 0.4806.

### Insights or Next Steps
*   The current segmentation pipeline produces a relatively low Dice score for the best-identified threshold (0.4806), indicating that the segmentation accuracy could be significantly improved. Further refinement of the pipeline (e.g., exploring different vesselness filters, morphological operations, or post-processing techniques) is recommended.
*   The sensitivity analysis should be extended to cover all images in the `synthetic_images` dataset, not just `sample_img`, to understand the generalizability of the optimal threshold and the pipeline's performance across different images.


# Task
Explain how to adjust the Frangi filter parameters, including `sigmas`, `alpha`, `beta`, `gamma`, and `black_ridges`, and describe their impact on vessel segmentation.

## Explain Frangi Filter Parameters

### Subtask:
Provide a detailed explanation of how to adjust the Frangi filter parameters, including `sigmas`, `alpha`, `beta`, `gamma`, and `black_ridges`, along with their impact on vessel segmentation.


## Summary:

### Data Analysis Key Findings

The Frangi filter parameters are crucial for optimizing vessel segmentation results. Each parameter influences how the filter identifies and enhances vessel-like structures based on local intensity and shape characteristics:

*   **`sigmas`**: This parameter defines the range of standard deviations for the Gaussian filters applied. It dictates the scale of structures the filter is sensitive to. A smaller range of `sigmas` detects finer, smaller vessels, while a larger range targets thicker, larger vessels. Properly setting this range is essential to match the expected vessel diameters in the image.
*   **`alpha`**: Related to `R_A`, this parameter controls the filter's sensitivity to plate-like structures. Higher `alpha` values make the filter more tolerant to sheet-like features, potentially leading to the inclusion of non-vessel noise, while lower values increase strictness in identifying line-like structures, reducing false positives from sheet-like shapes. A common setting is 0.5.
*   **`beta`**: Related to `R_B`, this parameter governs the filter's sensitivity to blob-like structures. A higher `beta` allows for greater tolerance of blob-like noise, whereas a lower `beta` enforces stricter detection of line-like structures, minimizing false positives caused by blobs. A typical value is 15.
*   **`gamma`**: This parameter, related to `S^2`, determines the filter's sensitivity to the overall strength or intensity of structures. Increasing `gamma` requires stronger vessel-like features to be detected, effectively suppressing weaker structures and noise. Conversely, a lower `gamma` will detect fainter structures but may also include more noise. A common setting is 100.
*   **`black_ridges`**: This boolean flag specifies the expected intensity relationship between vessels and their background. When set to `True`, the filter seeks dark ridges on a bright background (e.g., blood vessels in an angiogram). When `False`, it looks for bright ridges on a dark background. Setting this correctly is fundamental for accurately identifying the polarity of the desired features.

### Insights or Next Steps

*   **Iterative Tuning**: Effective vessel segmentation often requires iterative tuning of these parameters, especially `sigmas`, `alpha`, `beta`, and `gamma`, alongside visual inspection of the output, to achieve optimal detection of vessels across varying scales and contrasts.
*   **Contextual Parameter Selection**: The optimal values for `alpha`, `beta`, and `gamma` are often dataset-dependent and might require experimentation. Starting with typical values (e.g., `alpha=0.5`, `beta=15`, `gamma=100`) and adjusting based on the specific image characteristics and desired vessel appearance is a good practice.
